# Setup and Load Data

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
# df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# # Checkpoint as parquet (run once, then comment out)
# TEAM_DIR = "dbfs:/student-groups/Group_3_2"
# dbutils.fs.mkdirs(TEAM_DIR)
# df_otpw.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
#df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
df = spark.read.parquet(f"{TEAM_DIR}/otpw_12m.parquet")

In [0]:
display(df)

# Correlation

In [0]:
columns = [
    "DEP_DEL15",
    "DISTANCE",
    "CRS_ELAPSED_TIME",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyWindDirection",
    "HourlyVisibility",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyStationPressure",
    "HourlySeaLevelPressure"
]

df_casted = df.select([col(c).cast("double").alias(c) for c in columns])


In [0]:
from pyspark.sql.functions import when

df_cleaned = df.select([
    when(col(c).isin("", "NA", "null"), None)
    .otherwise(col(c))
    .cast("double")
    .alias(c)
    for c in columns
]).dropna()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(inputCols=columns, outputCol="features")
df_vector = assembler.transform(df_cleaned)

corr_matrix = Correlation.corr(df_vector, "features", method="spearman").head()[0]

corr_matrix.toArray()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

corr_array = corr_matrix.toArray()
corr_df = pd.DataFrame(corr_array, index=columns, columns=columns)

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5
)
plt.title("Spearman Correlation Heatmap")
plt.show()

# General Info and Feature Types

In [0]:
df.printSchema()

In [0]:
# Dataset dimensions
num_rows = df.count()
num_cols = len(df.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

# Handle Duplicates and Missing Data

## Duplicates

In [0]:
total = df.count()
distinct = df.dropDuplicates().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicate rows: {total - distinct}")

## Missing Data for Response Variable

In [0]:
null_exprs = [
    count(
        when(col(c).isNull() | (col(c).cast("string") == ""), c)
    ).alias(c)
    for c in df.columns
]
null_counts = df.select(null_exprs).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_percentage"] = (null_counts["null_count"] / num_rows * 100).round(2)

null_counts = (
    null_counts
    .reset_index()
    .rename(columns={"index": "column_name"})
    .sort_values("null_percentage", ascending=False)
)

high_missing_cols = null_counts[null_counts["null_percentage"] > 50]
print(f"Columns with >50% missing: {len(high_missing_cols)}")
display(null_counts)

In [0]:
delayed = df.filter(col("DEP_DEL15") == 1)
on_time = df.filter(col("DEP_DEL15") == 0)
missing = df.filter(col("DEP_DEL15").isNull())

print(f"Total delayed flights: {delayed.count()}")
print(f"Total on-time flights: {on_time.count()}")
print(f"Missing DEP_DEL15: {missing.count()}")

We can see that DEP_DEL15, the variable we are interested in, has 3.02 percent missing data. This could be due to the fact that there are cancelled flights that may have null DEP_DEL15 values.

In [0]:
# Is it true that all cancelled flights will not have DEP_DELAY and DEP_DEL15?
cancelled = df.filter(col("CANCELLED") == 1)
total_cancelled = cancelled.count()
cancelled_with_delay = cancelled.filter(col('DEP_DELAY').isNotNull()).count()
cancelled_with_del15 = cancelled.filter(col('DEP_DEL15').isNotNull()).count()

print(f"Total cancelled flights: {total_cancelled} ({total_cancelled / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DELAY: {cancelled_with_delay} ({cancelled_with_delay / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DEL15: {cancelled_with_del15} ({cancelled_with_del15 / num_rows * 100:.2f}% of dataset)")

We decide to remove cancelled flights as it may introduce confounding variables on an already unbalanced dataset.

In [0]:
# Filter out cancelled flights
df = df.filter(col("CANCELLED") == 0)
num_rows = df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
missing = df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing.count()}")

There are no more missing data for DEP_DEL15.

In [0]:
# Class balance of DEP_DEL15
target_dist = df.groupBy("DEP_DEL15").count().toPandas()
target_dist["percentage"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(2)
print(target_dist)

plt.figure(figsize=(6, 4))
plt.bar(target_dist["DEP_DEL15"].astype(str), target_dist["count"])
plt.title("DEP_DEL15 Class Distribution (3-Month OTPW)")
plt.xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
plt.ylabel("Count")
for i, row in target_dist.iterrows():
    plt.text(i, row["count"], f"{row['percentage']}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Other missing data explaination

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

We first check report type for missing data of weather. One of the reasons why there are so many missing values is that the weather dataset has multiple report types (FM-15, FM-16, FM-12, etc) that do not contain all information. For example:

* FM-15: Standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* FM-16: Special weather report issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* FM-12: Synoptic report issued every 6–12 hours with fewer variables; many fields are missing and it does not align well with hourly flight data.
* SOD (Summary of Day): Daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* SHEF and SY-MT: Rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

Because of this, we focus primarily on FM-15 for modeling. In the future we can optionally include FM-16 if we want to capture extreme weather events. As for the other report tyes, we will and drop them to ensure consistent and reliable features for each flight.

In [0]:
df.groupBy("REPORT_TYPE") \
  .agg(count("*").alias("count")) \
  .display()

In [0]:
df = df.filter(col("REPORT_TYPE") == "FM-15")
num_rows = df.count()
print(f"After filtering FM-15: {num_rows:,} rows remaining")

Next, we look into source type, since the weather dataset is based off of many different sources, there may be sources that have more missing data than others.

In [0]:
df.groupBy("SOURCE").count().display()

In [0]:
weather_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyPresentWeatherType",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    "HourlySkyConditions",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

exprs = [
    (count(when(col(c).isNull(), True)) / count("*")).alias(f"{c}_missing_pct")
    for c in weather_cols
]

df.groupBy("SOURCE").agg(*exprs).display()

In the table, we can see that source 7 and 6 make up the majority of the dataset. There are missing data, but there are also a lot of columns that have are filled out in the majority of rows.

Additionally based off of the data dictionary to the original data, null values represent zero or not observed, and M represetns missing data. 

In the OTPW dataset, there are both 0 and null values in precipitation, but we believe that is just an artifact of merging the data. Specifically when the report type is FM-15 (regular at hourly intervals), we should treat nulls as 0s.

In [0]:
from pyspark.sql.types import FloatType

weather_continuous_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    #"HourlyPresentWeatherType",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    #"HourlySkyConditions",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

for c in weather_cols:
    df = df.withColumn(c, col(c).cast(FloatType()))
    if  c in {"HourlyPrecipitation"}:
        df = df.na.fill({c: 0})  # fill nulls column by column

In [0]:
df.display(100)

# Normalization and Transformations

Based off of our [EDA Summary](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/3940281024316733?o=4021782157704243#command/7866244608655161), we can see that precipitation, humidity, wind speed, visibility and temperature are the strongest flight delay signals. We will use them in our modeling, and they will need to be normalized and transformed.

In [0]:
# Collect data to pandas (small enough sample!)
precip_pd = df.select("HourlyPrecipitation").sample(0.1).toPandas()  # sample 10% to speed up

plt.figure(figsize=(8,5))
plt.hist(precip_pd['HourlyPrecipitation'], bins=50)
plt.xlabel('Hourly Precipitation')
plt.ylabel('Count')
plt.title('Distribution of Hourly Precipitation')
plt.show()

In [0]:
import numpy as np
from pyspark.sql.functions import log1p

plt.hist(precip_pd['HourlyPrecipitation'].apply(lambda x: np.log1p(x)), bins=50)
plt.xlabel('Log(Precipitation + 1)')
plt.ylabel('Count')
plt.title('Log-transformed Hourly Precipitation')
plt.show()

Precipitation is heavily skewed, even after log transformations it is still skewed. A better option would be to turn it into a binary feature where 0 = no precipitation and 1 = precipitation

In [0]:
df = df.withColumn(
    "PrecipBinary",
    when(col("HourlyPrecipitation") > 0, 1).otherwise(0)
)

In [0]:
# Sample data to pandas (optional for large datasets)
sample_pd = df.select("PrecipBinary").sample(0.1, seed=42).toPandas()

# Count occurrences of 0 and 1
counts = sample_pd["PrecipBinary"].value_counts().sort_index()

# Plot bar chart
plt.figure(figsize=(6,4))
plt.bar(counts.index.astype(str), counts.values)
plt.xticks([0,1])
plt.xlabel("Precipitation Binary (0 = No Rain, 1 = Any Rain)")
plt.ylabel("Count")
plt.title("Distribution of Binary Precipitation")
plt.show()

Next we look at the other numeric features: humidity, wind speed, visibility and temperature.

In [0]:
weather_numeric_cols = [
    "HourlyRelativeHumidity",
    "HourlyWindSpeed",
    "HourlyVisibility",
    "HourlyDryBulbTemperature"
]

sample_pd = df.select(weather_numeric_cols).sample(0.1, seed=42).toPandas()

plt.figure(figsize=(16,10))

for i, col_name in enumerate(weather_numeric_cols):
    plt.subplot(2,2,i+1)
    plt.hist(sample_pd[col_name], bins=50)
    plt.title(f'Distribution of {col_name}')
    plt.xlabel(col_name)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

Relative humidity and temperature seem to be relatively normal. We will just need to normalize them.

In [0]:
from pyspark.sql.functions import mean, stddev

for c in {"HourlyRelativeHumidity", "HourlyDryBulbTemperature"}:
    stats = df.select(mean(c).alias("mean"), stddev(c).alias("std")).collect()[0]
    df = df.withColumn(c + "_scaled", (col(c) - stats["mean"]) / stats["std"])

In [0]:
# Columns to plot
scaled_cols = ["HourlyRelativeHumidity_scaled", "HourlyDryBulbTemperature_scaled"]

# Sample 10% to avoid memory issues
sample_pd = df.select(scaled_cols).sample(0.1, seed=42).toPandas()

# Plot histograms
plt.figure(figsize=(12,5))

for i, col_name in enumerate(scaled_cols):
    plt.subplot(1, 2, i+1)
    plt.hist(sample_pd[col_name], bins=50)
    plt.title(f'Distribution of {col_name}')
    plt.xlabel(col_name)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

Wind speed is right skewed. We will log transform it and then scale it.

In [0]:
# Log-transform (creates a new column)
df = df.withColumn("HourlyWindSpeed_log", log1p(col("HourlyWindSpeed")))

# Compute mean and std
stats = df.select(
    mean("HourlyWindSpeed_log").alias("mean"),
    stddev("HourlyWindSpeed_log").alias("std")
).collect()[0]

# Scale to mean=0, std=1
df = df.withColumn(
    "HourlyWindSpeed_scaled",
    (col("HourlyWindSpeed_log") - stats["mean"]) / stats["std"]
)

In [0]:
# Sample 10% to avoid memory issues
sample_pd = df.select("HourlyWindSpeed_scaled").sample(0.1, seed=42).toPandas()

# Plot histogram
plt.figure(figsize=(8,5))
plt.hist(sample_pd["HourlyWindSpeed_scaled"], bins=50)
plt.xlabel("HourlyWindSpeed_scaled")
plt.ylabel("Count")
plt.title("Histogram of Scaled Log-Transformed Wind Speed")
plt.show()

Visibility is sparce and left skewed, with a max. We decide to make it into a categorical variable instead.

| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "VisibilityCat",
    when(col("HourlyVisibility") < 2, 0)
    .when((col("HourlyVisibility") >= 2) & (col("HourlyVisibility") < 5), 1)
    .when((col("HourlyVisibility") >= 5) & (col("HourlyVisibility") < 10), 2)
    .otherwise(3)
)

In [0]:
# Sample to Pandas (optional for large datasets)
sample_pd = df.select("VisibilityCat").sample(0.1, seed=42).toPandas()

# Count occurrences of each category
counts = sample_pd["VisibilityCat"].value_counts().sort_index()

# Plot bar chart
plt.figure(figsize=(6,4))
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Visibility Category (0=Very Low, 3=High)")
plt.ylabel("Count")
plt.title("Distribution of Visibility Categories")
plt.xticks(counts.index)
plt.show()

# Feature Engineering and Creation

We will create two additional features, measuring if the flight is on a weekend, as well as and combination of the origin and destination airport.

In [0]:
df = df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

In [0]:
df.groupBy("IsWeekend").count().display()

In [0]:
from pyspark.sql.functions import concat_ws, col

df = df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)

We will need to encode this new variable for ML models

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Step 1: Convert Route string into numeric index
route_indexer = StringIndexer(inputCol="Route", outputCol="RouteIndex", handleInvalid="keep")
df = route_indexer.fit(df).transform(df)

# Step 2: One-hot encode the index
route_encoder = OneHotEncoder(inputCols=["RouteIndex"], outputCols=["RouteVec"])
df = route_encoder.fit(df).transform(df)

### Summary

Focused on the 5 most important weather variables:
* Precipitation (PrecipBinary): 1 if rain, 0 if no rain
  * this was heavily skewed so i decided to make it binary
* Humidity (HourlyRelativeHumidity_scaled): nomalized (mean 0, std 1)
* Temperature (HourlyDryBulbTemperature_scaled): nomalized (mean 0, std 1)
* Wind Speed (HourlyWindSpeed_scaled): log+1 transformed and nomalized (mean 0, std 1)
* Visibility (VisibilityCat):
| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

I also created additional features such as IsWeekend and Route. Route combines origin and destination and is one hot encoded for ML modeling.

The notebook also includes steps taken to filter out cancelled fights and to only use FM-15 data.

# Future Plans

TODO:
* Specify the feature transformations for the pipeline and justify these features given the target (e.g., hashing trick, tf-idf, stopword removal, lemmatization, tokenization, etc..)
* Other feature engineering efforts, i.e., interaction terms, Breiman's method, etc.

We are concerned with false negatives. This is the case where we predict dep_delay15 is 0 (no delay) when there is actually a delay (1). Therefore we will concern ourselves with measuring recall. We will avoid accuracy since the dataset is imbalanced; there are a lot more non-delayed flights then there are delayed ones.